[![Text: CC BY-SA 4.0](https://img.shields.io/badge/text-CC%20BY--SA%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by-sa/4.0/)
[![Code: MIT](https://img.shields.io/badge/code-MIT-yellow.svg)](https://opensource.org/license/mit)

This notebook is part of course materials for CS 545: Machine Learning at
Colorado State University. These notebooks are based on the PyTorch version of
[Dive into Deep Learning](https://d2l.ai) by Aston Zhang, Zachary C. Lipton,
Mu Li and Alexander J. Smola, used under
[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) and heavily
modified by [Asa Ben-Hur](https://www.cs.colostate.edu/~asa/) with
[Claude](https://claude.com) AI. The text is released under
[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/); the code is
released under the [MIT license](https://opensource.org/license/mit).

# Pooling

Our ultimate question about an image is usually global — *does it contain a cat?* — so the units of the final layer must be sensitive to the entire input.  But convolutional layers are local by construction, and we saw in notebook 2 how slowly the receptive field grows with depth: a stack of $3 \times 3$ layers gains only two pixels of reach per layer.

The way out is to **reduce spatial resolution as we go**.  Coarser feature maps mean each kernel covers a larger fraction of the original image, so the receptive field grows multiplicatively rather than additively, and by the last layer a unit can see everything.

This notebook covers **pooling**, the classical mechanism for doing this.  Pooling serves two purposes at once: it downsamples, and it makes the representation somewhat insensitive to exactly *where* a feature was found.

In [ ]:
import torch
from torch import nn

## Maximum pooling and average pooling

Like a convolutional layer, a **pooling** operator slides a fixed-shape window over the input and produces one output per position.  Unlike a convolutional layer it has **no parameters** — no kernel to learn.  It simply computes the maximum or the mean of the values in the window, giving **max-pooling** and **average pooling** respectively.

Average pooling is as old as CNNs, and the idea is just careful downsampling: rather than keeping every second pixel, average over adjacent ones, which combines the information and improves the signal-to-noise ratio.  Max-pooling arrived later, introduced by Riesenhuber and Poggio (1999) in cognitive neuroscience as a model of how information might be aggregated hierarchically for object recognition.  In practice max-pooling is almost always the better choice, for a reason worth stating: a convolutional layer produces a *large* response where a feature is present, and taking the maximum preserves that response, whereas averaging dilutes it with the surrounding non-responses.

The mechanics are the same as cross-correlation: start at the upper left, slide left to right and top to bottom, and at each position compute the maximum (or mean) of the input elements under the window.

In [1]:
# this figure is drawn rather than linked, so the notebook needs no network
import matplotlib
from matplotlib import pyplot as plt
matplotlib.rcParams['svg.fonttype'] = 'none'   # keep text as text in the SVG
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('svg')              # crisp vector output
except ImportError:
    pass

# Shared idiom for the grid figures of this module.  Geometry is in inches and
# the axes are set so one data unit is one inch, so text never outgrows a cell.
CELL = 0.34
SHADE = '#cfe0ef'          # cells taking part in the highlighted computation
SHADE2 = '#f3ddd0'         # a second, distinct computation
PAD_FILL = '0.955'         # padding added around the input
FS = 7.6

def _grid(ax, x0, y0, values, shade=(), shade2=(), pad=(), cell=CELL, fs=FS,
          colors=None, z=2):
    """Draw a matrix with its top-left cell's top-left corner at (x0, y0).

    `z` sets the drawing order, so overlapping channel slices occlude cleanly.
    """
    from matplotlib.patches import Rectangle
    shade, shade2, pad = set(shade), set(shade2), set(pad)
    colors = colors or {}
    for r, row in enumerate(values):
        for c, v in enumerate(row):
            fill = colors.get((r, c),
                              SHADE if (r, c) in shade else
                              SHADE2 if (r, c) in shade2 else
                              PAD_FILL if (r, c) in pad else 'white')
            ax.add_patch(Rectangle((x0 + c * cell, y0 - (r + 1) * cell),
                                   cell, cell, facecolor=fill,
                                   edgecolor='0.5', linewidth=0.8, zorder=z))
            if v is not None:
                ax.text(x0 + (c + 0.5) * cell, y0 - (r + 0.5) * cell, str(v),
                        ha='center', va='center', fontsize=fs,
                        color='0.55' if (r, c) in pad else 'black', zorder=z + 1)
    return len(values[0]) * cell, len(values) * cell


def _label(ax, x, y, text, fs=8.2, weight='normal'):
    ax.text(x, y, text, ha='center', va='center', fontsize=fs, fontweight=weight)


def _op(ax, x, y, symbol):
    ax.text(x, y, symbol, ha='center', va='center', fontsize=12, color='0.35')


def _finish(fig, ax, x0, x1, y0, y1):
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.axis('off')
    fig.set_size_inches(x1 - x0, y1 - y0)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)


def _row_layout(panels, gap=0.42, top=0.0):
    """Left-to-right x positions for a row of (width, height) panels."""
    xs, cursor = [], 0.0
    for w, h in panels:
        xs.append(cursor)
        cursor += w + gap
    return xs, cursor - gap


# ---------------------------------------------------------------- 03_02 / 03_03

def pooling_figure():
    """Max pooling: the same sliding window, but no parameters and no sum."""
    X = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
    Y = [[4, 5], [7, 8]]
    fig, ax = plt.subplots()
    top, x = 0.0, 0.0
    w1, h1 = _grid(ax, x, top, X, shade=[(0, 0), (0, 1), (1, 0), (1, 1)])
    _label(ax, x + w1 / 2, top + 0.20, 'input')
    x1 = x + w1
    ax.text(x1 + 0.46, top - h1 / 2, '2x2\nmax pool', ha='center', va='center',
            fontsize=8, color='0.35', linespacing=1.4)
    x2 = x1 + 0.92
    _op(ax, x2 + 0.15, top - h1 / 2, '=')
    x3 = x2 + 0.40
    w3, h3 = _grid(ax, x3, top - (h1 - 2 * CELL) / 2, Y, shade=[(0, 0)])
    _label(ax, x3 + w3 / 2, top + 0.20, 'output')
    ax.text((x3 + w3) / 2, top - h1 - 0.28, r'$\max(0, 1, 3, 4) = 4$',
            ha='center', va='center', fontsize=8.4)
    _finish(fig, ax, -0.18, x3 + w3 + 0.18, top - h1 - 0.50, top + 0.40)

pooling_figure()

The output above has height and width 2, its elements being

$$
\max(0, 1, 3, 4)=4,\;
\max(1, 2, 4, 5)=5,\;
\max(3, 4, 6, 7)=7,\;
\max(4, 5, 7, 8)=8.
$$

More generally, a $p \times q$ pooling layer aggregates over a region of that size.

Now the invariance claim, made concrete with the edge detector from notebook 2.  Feed the convolutional layer's output into $2 \times 2$ max-pooling and call the result `Y`.  Whether the edge showed up at `X[i, j]`, `X[i, j+1]`, `X[i+1, j]` or `X[i+1, j+1]`, the pooling layer reports `Y[i, j] = 1`.  The layer still detects the pattern, but it has stopped caring about a displacement of one pixel.

The implementation is `corr2d` with the multiply-and-sum replaced by a max or a mean:

In [ ]:
def pool2d(X, pool_size, mode='max'):
    """2D pooling over a single-channel input, stride 1, no padding."""
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j] = X[i: i + p_h, j: j + p_w].max()
            elif mode == 'avg':
                Y[i, j] = X[i: i + p_h, j: j + p_w].mean()
    return Y

Using the input from the figure:

In [ ]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
Y = pool2d(X, (2, 2))
print(Y)
assert torch.allclose(Y, torch.tensor([[4.0, 5.0], [7.0, 8.0]]))

In [ ]:
pool2d(X, (2, 2), 'avg')

Now the translation-invariance claim, run rather than asserted.  We take the vertical-edge image from notebook 2, shift it by one pixel, and compare what the convolutional layer reports with what the pooled output reports:

In [ ]:
def corr2d(X, K):
    """Compute the 2D cross-correlation of input X with kernel K (notebook 2)."""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

K = torch.tensor([[1.0, -1.0]])              # the vertical-edge detector

image = torch.ones((6, 8)); image[:, 2:6] = 0        # edges at columns 2 and 6
shifted = torch.ones((6, 8)); shifted[:, 3:7] = 0    # the same, moved right by 1

conv_out, conv_shifted = corr2d(image, K), corr2d(shifted, K)
pool_out = pool2d(conv_out, (1, 2))
pool_shifted = pool2d(conv_shifted, (1, 2))

print('conv output, row 0:        ', conv_out[0].tolist())
print('conv output shifted, row 0:', conv_shifted[0].tolist())
print('pooled, row 0:             ', pool_out[0].tolist())
print('pooled shifted, row 0:     ', pool_shifted[0].tolist())
print()
print('positions where conv outputs differ:  ',
      int((conv_out != conv_shifted).sum()))
print('positions where pooled outputs differ:',
      int((pool_out != pool_shifted).sum()))

Both representations move when the image moves — pooling does not make the network blind to position, and we would not want it to.  What it does is *blur* the response: the pooled maps of the original and shifted image disagree in fewer places, because a one-pixel displacement inside a window leaves the maximum unchanged.  Stack several pooling layers and a feature can wander several pixels before the deep representation notices.

## Padding and stride

Pooling layers change the output shape, and as with convolution we control that with padding and stride.  PyTorch's built-in `nn.MaxPool2d` works on rank-4 tensors of shape `(batch, channels, height, width)`, so we construct an input with a batch size and channel count of 1:

In [ ]:
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
X

One difference from convolution is worth flagging, because it surprises people.  Since pooling exists to *aggregate*, PyTorch defaults the stride to the window size rather than to 1 — a `(3, 3)` window gets a stride of `(3, 3)`, so the windows tile the input rather than overlapping it.  A `nn.MaxPool2d(3)` on a $4 \times 4$ input therefore yields a single number:

In [ ]:
pool = nn.MaxPool2d(3)
pool(X)

Both can of course be specified explicitly:

In [ ]:
pool = nn.MaxPool2d(3, padding=1, stride=2)
pool(X)

And the window may be rectangular, with its own padding and stride per axis:

In [ ]:
pool = nn.MaxPool2d((2, 3), stride=(2, 3), padding=(0, 1))
pool(X)

### Checking `pool2d` against PyTorch

Our `pool2d` uses stride 1 and no padding, so the comparison needs `stride=1` on the PyTorch side:

In [ ]:
X2 = torch.rand(7, 9)
for mode, layer in [('max', nn.MaxPool2d(2, stride=1)),
                    ('avg', nn.AvgPool2d(2, stride=1))]:
    ours = pool2d(X2, (2, 2), mode)
    theirs = layer(X2.reshape(1, 1, *X2.shape)).reshape(ours.shape)
    assert torch.allclose(ours, theirs, atol=1e-6), mode
    print(f'{mode}-pooling matches, output shape {tuple(ours.shape)}')

## Multiple channels

With multi-channel input, a pooling layer pools **each channel separately** rather than summing over channels as a convolutional layer does.  So the number of output channels equals the number of input channels — pooling changes spatial resolution and nothing else.

Concatenating `X` and `X + 1` along the channel axis gives a two-channel input:

In [ ]:
X = torch.cat((X, X + 1), 1)
print(X.shape)
X

In [ ]:
pool = nn.MaxPool2d(3, padding=1, stride=2)
out = pool(X)
print(out.shape, ' # channels unchanged')
out

## A note on pooling today

Pooling is a fixed, parameter-free operation, and once GPUs made large networks affordable, architects began asking whether the network should choose its own downsampling instead.  Many modern architectures replace pooling with **strided convolutions**, which downsample with learned weights, and most end with **global average pooling** — an average over the entire feature map, collapsing each channel to a single number — in place of a large fully connected layer.

None of this makes pooling obsolete; max-pooling remains a standard component, and it is a part of LeNet, which we build next.  But it is worth knowing that the downsampling *role* is essential while this particular way of filling it is a design choice.

## Summary

* Pooling aggregates values over a window.  It has **no parameters**, and all the convolution semantics — window size, padding, stride — carry over unchanged.
* Pooling is applied to each channel independently, so it leaves the number of channels alone.
* Its two purposes are **downsampling**, which accelerates the growth of the receptive field, and **partial translation invariance**: a small displacement inside a window does not change the maximum.
* Max-pooling is generally preferred to average pooling, because it preserves the strong responses that indicate a feature is present rather than diluting them.
* A $2 \times 2$ window with stride 2 is the common choice, quartering the spatial resolution.
* PyTorch defaults a pooling layer's stride to its window size — unlike a convolution, whose stride defaults to 1.

## Exercises

1. How would you implement average pooling using a convolution?  Write the kernel explicitly and verify against `nn.AvgPool2d`.
2. Can you implement max-pooling as a convolution?  Why or why not?
3. What is the computational cost of a pooling layer, in terms of input size $c \times h \times w$, window $p_\textrm{h} \times p_\textrm{w}$, and stride?  How does it compare with the convolutional layer that feeds it?
4. Why do you expect max-pooling and average pooling to work differently?  Construct an input where they give very different answers.
5. Do we need a separate minimum-pooling layer?  Could you replace it with something else?
6. Consider the softmax operation as an alternative aggregation: $\mathrm{softmax}(\mathbf{x}) = \sum_i \exp(\lambda x_i) x_i / \sum_i \exp(\lambda x_i)$.  What does it converge to as $\lambda \to \infty$?  As $\lambda \to 0$?  Why might this not be a popular choice in practice?
7. Rerun the shift experiment in this notebook with average pooling instead of max-pooling, and with a $1 \times 4$ window.  How much displacement does each tolerate?